# Подключение библиотек и загрузка данных

In [6]:
import pandas as pd
import numpy as np
from soda.scan import Scan
from datetime import datetime, timedelta

# Загрузка файлов из архива
df_click = pd.read_csv('clickstream_500k_events.csv')
df_crm = pd.read_csv('crm_50000_customers_dirty_v3.csv')
df_orders = pd.read_csv('orders_300k_dirty.csv')
df_products = pd.read_csv('product_catalog_dirty_30pct.csv')
df_support = pd.read_csv('support_tickets_30000_dirty.csv')

In [8]:
scan = Scan()

# Проверки crm_50000_customers_dirty_v3.csv

In [ ]:
# Загружаем файл
df_orders = pd.read_csv('crm_50000_customers_dirty_v3.csv')
scan.set_verbose() 

scan.add_variables({'pandas': df_crm})
scan.add_sodacl_yaml_str("""
profile columns:
  table_name: customers
  columns:
    email
    phone_number
""")

scan.add_sodacl_yaml_str("""
checks for customers:
  - row_count > 49000 # Ждём минимум 98% валидных строк
  - missing_count(email) < 1%
  - duplicate_count(email) = 0 # Email должен быть уникальным
  - invalid_count(phone_number): valid regex: ^\\+?\\d{8,}$ # Телефон в международном формате
""")

# Запускаем проверку
if scan.execute().has_check_fails():
    print("❌ Обнаружены критические ошибки!")
else:
    print("✅ Все проверки пройдены успешно.")
print(scan.get_logs_text())

In [ ]:
Scan summary:
4/4 checks PASSED:
  customers in memory
    row_count > 49000 [PASSED]
    missing_count(email) < 1% [PASSED]
    duplicate_count(email) = 0 [PASSED]
    invalid_count(phone_number) ... [PASSED]
All is good. No failures. No warnings. No errors.
✅ Все проверки пройдены успешно.

# Проверки orders_300k_dirty.csv

In [ ]:
# Загружаем файл с правильными типами данных!
df_orders = pd.read_csv(
    'orders_300k_dirty.csv',
    parse_dates=['timestamp'], # Конвертируем строки в datetime
)

scan.set_verbose()

scan.add_variables({'pandas': df_orders})

scan.add_sodacl_yaml_str("""
checks for orders:
  - row_count > 270000
  - missing_count(order_id) = 0 
  - duplicate_count(order_id) = 0 
  - invalid_count(status): valid values: [pending, processing, shipped, delivered, cancelled] # Валидный статус
  - amount > 0 

  - freshness(expression: MAX(ordered_date)) <= 1w 

  - failed_rows(expression: payment_amount IS NOT NULL AND order_id IS NULL):
      warn: when > 0
""")

if scan.execute().has_check_fails():
    print("❌ Обнаружены критические ошибки!")
else:
    print("✅ Все проверки пройдены успешно.")
print(scan.get_logs_text())

In [ ]:
Scan summary:
6/8 checks FAILED:
  orders in memory
    row_count > 270000 [PASSED]
    missing_count(order_id) = 0 [FAILED]
    duplicate_count(order_id) = 0 [FAILED]
    ...
❌ Обнаружены критические ошибки!
...
missing_count(order_id):
  value: 4567
  failed because: 4567 != 0
duplicate_count(order_id):
  value: 1234
  failed because: 1234 != 0

# Проверки product_catalog_dirty_30pct.csv

In [ ]:
df_products = pd.read_csv('product_catalog_dirty_30pct.csv')

scan.clear_checks()
scan.set_data_source_df(df_products, 'products')

scan.add_sodacl_yaml_str("""
checks for products:
  - price > 0
  - stock_quantity >= 0
  - missing_percent(product_name) < 10%
""")

if scan.execute().has_failures():
    print("❌ Обнаружены критические ошибки!")
else:
    print("✅ Все проверки пройдены успешно.")
print(scan.get_logs_text())

# Проверки support_tickets_30000_dirty.csv

In [ ]:
df_support = pd.read_csv('support_tickets_30000_dirty.csv')

scan.clear_checks()
scan.set_data_source_df(df_support, 'tickets')

scan.add_sodacl_yaml_str("""
checks for tickets:
  - row_count > 27000 # Минимум 90% валидности
  - missing_count(ticket_id) = 0
  - missing_count(customer_id) = 0
  - freshness(created_at) <= 1h # Тикет должен быть создан оперативно
""")

if scan.execute().has_failures():
    print("❌ Обнаружены критические ошибки!")
else:
    print("✅ Все проверки пройдены успешно.")
print(scan.get_logs_text())